# NeurIPS Foundation v2: контекст, доверие и пересмотр уверенности

Этот notebook теперь оформлен как читаемый research narrative. Главная цель — не просто прогнать пайплайн, а последовательно показать, **какие данные у нас есть, где задачи вырождаются, где реально появляется signal, и зачем нужен context-aware encoder**.

## Что здесь является главной научной историей

Мы идём от простого к сильному:

1. Сначала проверяем, насколько силён сам рынок как baseline.
2. Затем отделяем **prediction** от **trust/meta-prediction**.
3. После этого смотрим на **repricing** как на основную динамическую задачу.
4. Наконец, проверяем, даёт ли **retrieved context** хоть какой-то честный выигрыш.

Перед каждой важной таблицей я теперь коротко пишу, **зачем она нужна** и **на какие колонки смотреть**.

## Импорты и окружение

Стандартный стек: `pandas`, `sklearn`, `seaborn`, plus локальные benchmark helpers. Важно, что Polymarket `crypto` domain здесь **не используется**: он исключён из анализа как неполный и шумный. При этом `BTC/ETH` сохраняются как **внешние сигналы**, а не как Polymarket-домен.

In [2]:
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from polymarket_research import PolymarketDataset
from polymarket_research.utils import (
    build_family_id,
    clipped_probabilities,
    parse_listish,
    safe_auc,
    safe_average_precision as safe_ap,
    setup_root,
    tag_jaccard,
)

REPO_ROOT = setup_root()

from benchmarks.benchmark_utils import (
    add_time_features,
    build_multi_horizon_terminal_dataset,
    build_repricing_dataset,
    rolling_time_splits,
)
from benchmarks.covariate_utils import (
    asof_join_covariates,
    load_external_covariates,
    pivot_covariates_to_wide,
)


## Конфигурация и вспомогательные функции

Здесь задаются домены, горизонты задач и вспомогательные функции. Идейно важны три вещи:

- `DOMAINS`: какие домены реально входят в paper spine;
- `TERMINAL_HORIZONS` и `REPRICING_FUTURE_HOURS`: что именно мы называем forecasting и repricing;
- family/context helpers: как мы строим weakly supervised связи между родственными рынками.

In [2]:
DOMAINS = ('politics', 'geopolitics', 'technology', 'finance_economy')
MIN_PROBABILITY_ROWS = 288
MAX_SNAPSHOT_STALENESS_HOURS = 12.0

TERMINAL_HORIZONS = (24, 72, 168)
REPRICING_FUTURE_HOURS = 24
REPRICING_LOOKBACK_HOURS = 24
REPRICING_SAMPLE_EVERY_HOURS = 12
REPRICING_MOVE_THRESHOLD = 0.15

SHOCK_Z_THRESHOLD = 2.0
SHOCK_STD_WINDOW = 288
SHOCK_MAX_AGE = '2D'
EXTERNAL_PATH = REPO_ROOT / 'cached_data' / 'external_covariates'

RETRIEVAL_MAX_MARKETS_PER_DOMAIN = 90
TOP_K = 5
RETRIEVED_CONTEXT_COLS = ['retrieved_count', 'retrieved_prob_mean', 'retrieved_prob_std', 'retrieved_prob_gap', 'retrieved_trade_share']


## Явные assumptions и ограничения текущего протокола

Ниже перечислены ключевые design choices, которые иначе были бы спрятаны в конфигурации.

- `DOMAINS = politics, geopolitics, technology, finance_economy`
  Ограничиваемся полноценно загруженными и относительно чистыми доменами. Это значит, что выводы пока не претендуют на весь universe Polymarket.
- `MIN_PROBABILITY_ROWS = 288`
  Отсекаем совсем короткие и плохо наблюдаемые рынки. В результате cold-start и thin-market режимы представлены слабее.
- `MAX_SNAPSHOT_STALENESS_HOURS = 12`
  Terminal snapshot считаем валидным только если рядом с cutoff действительно есть наблюдение. Это уменьшает leakage-like artefacts от слишком редких рынков.
- `TERMINAL_HORIZONS = 24, 72, 168`
  Берём интерпретируемые горизонты до resolution. Всё, что происходит на более ранних или более коротких таймскейлах, пока вне этого notebook.
- `REPRICING_FUTURE_HOURS = 24`
  Определяем repricing как сдвиг вероятности на фиксированном горизонте вперёд. Это operational definition, а не единственно правильная постановка.
- `REPRICING_MOVE_THRESHOLD = 0.15`
  Large repricing задаётся порогом в 15 п.п. Это разумный, но всё же частично произвольный threshold, который дальше стоит проверять на sensitivity.
- `REPRICING_SAMPLE_EVERY_HOURS = 12`
  Сэмплируем не каждый snapshot, а раз в 12 часов, чтобы уменьшить сильную автокорреляцию. Цена за это — потеря части высокочастотной динамики.
- `SHOCK_Z_THRESHOLD = 2.0`
  Внешний shock по BTC/ETH определяется через rolling z-score. Это stylized detector, а не causal identification.
- `RETRIEVAL_MAX_MARKETS_PER_DOMAIN = 90`
  Retrieval пока строится не на полном market universe, а на ограниченном подмножестве, чтобы pairwise similarity оставалась вычислимой.
- `TOP_K = 5`
  В retrieved context берём только пять соседей. Это делает контекст компактным, но может терять часть релевантных рынков.


## Локальный класс для panel building

Эта логика пока остаётся внутри notebook, потому что research protocol ещё двигается. Класс отвечает только за сборку `markets`, `terminal` и `repricing` панелей из parquet-датасета.


In [ ]:
@dataclass(frozen=True)
class MarketPanelBuilder:
    """Build terminal and repricing panels from the base Polymarket dataset."""

    domains: tuple[str, ...]
    terminal_horizons: tuple[int, ...]
    min_probability_rows: int = 288
    max_snapshot_staleness_hours: float = 12.0
    repricing_future_hours: int = 24
    repricing_lookback_hours: int = 24
    repricing_sample_every_hours: int = 12
    repricing_move_threshold: float = 0.15

    def prepare_markets(self, markets_df: pd.DataFrame) -> pd.DataFrame:
        """Normalize market metadata and add a weak family identifier."""
        out = markets_df.copy()
        out['market_id'] = out['market_id'].astype(str)
        out['created_at'] = pd.to_datetime(out['created_at'], utc=True, errors='coerce')
        out['end_date'] = pd.to_datetime(out['end_date'], utc=True, errors='coerce')
        out['primary_domain'] = out['primary_domain'].fillna('unknown').astype(str)
        out['domain'] = out['primary_domain']
        out = out.dropna(subset=['market_id', 'created_at', 'end_date', 'final_yes_probability']).reset_index(drop=True)
        out = out.loc[out['domain'].isin(self.domains)].copy()
        out['family_id'] = [
            build_family_id(question, domain, tags)
            for question, domain, tags in zip(out['question'], out['primary_domain'], out['tag_labels'], strict=False)
        ]
        return out

    def prepare_probabilities(self, dataset: PolymarketDataset, markets_df: pd.DataFrame) -> pd.DataFrame:
        """Filter probability history down to the selected market universe."""
        probabilities = dataset.probabilities.copy()
        probabilities['market_id'] = probabilities['market_id'].astype(str)
        probabilities['timestamp_utc'] = pd.to_datetime(probabilities['timestamp_utc'], utc=True, errors='coerce')
        probabilities = probabilities.loc[probabilities['market_id'].isin(markets_df['market_id'])]
        return probabilities.sort_values(['market_id', 'timestamp_utc'], kind='stable').reset_index(drop=True)

    @staticmethod
    def latest_snapshot_before(probabilities_df: pd.DataFrame, market_id: str, cutoff: pd.Timestamp):
        """Fetch the latest market snapshot at or before a cutoff timestamp."""
        panel = probabilities_df.loc[(probabilities_df['market_id'] == market_id) & (probabilities_df['timestamp_utc'] <= cutoff)]
        if panel.empty:
            return None
        return panel.iloc[-1]

    @staticmethod
    def add_domain_dummies(df: pd.DataFrame, source_col: str = 'primary_domain') -> pd.DataFrame:
        """Append one-hot domain columns while keeping the original frame intact."""
        dummies = pd.get_dummies(df[source_col], prefix='domain', dtype=float)
        return pd.concat([df.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)

    def build_family_context_features(self, panel_df: pd.DataFrame, probabilities_df: pd.DataFrame, market_meta: pd.DataFrame, *, time_col: str, prob_col: str) -> pd.DataFrame:
        """Aggregate contemporaneous same-family probabilities into compact features."""
        family_map = market_meta.groupby('family_id')['market_id'].apply(list).to_dict()
        market_to_family = market_meta.set_index('market_id')['family_id'].to_dict()
        rows = []
        for row in panel_df[['market_id', time_col, prob_col]].itertuples(index=False):
            cutoff = getattr(row, time_col)
            base_prob = getattr(row, prob_col)
            family_id = market_to_family.get(row.market_id)
            related_ids = [market_id for market_id in family_map.get(family_id, []) if market_id != row.market_id]
            related_probs = []
            for related_id in related_ids:
                snapshot = self.latest_snapshot_before(probabilities_df, related_id, cutoff)
                if snapshot is None:
                    continue
                related_probs.append(float(snapshot['yes_probability']))
            rows.append({
                'market_id': row.market_id,
                time_col: cutoff,
                'family_related_count': float(len(related_probs)),
                'family_prob_mean': float(np.mean(related_probs)) if related_probs else np.nan,
                'family_prob_gap': float(np.max(related_probs) - np.min(related_probs)) if related_probs else np.nan,
                'family_vs_market_gap': float(abs(np.mean(related_probs) - base_prob)) if related_probs else np.nan,
            })
        return pd.DataFrame(rows)

    def build_terminal_panel(self, markets_df: pd.DataFrame, probabilities_df: pd.DataFrame) -> pd.DataFrame:
        """Build the terminal forecasting panel and append family/domain context."""
        terminal = build_multi_horizon_terminal_dataset(
            markets_df,
            probabilities_df,
            horizons_hours=self.terminal_horizons,
            max_snapshot_staleness_hours=self.max_snapshot_staleness_hours,
        )
        terminal = add_time_features(terminal)
        terminal = terminal.merge(markets_df[['market_id', 'primary_domain', 'family_id']], on='market_id', how='left')
        context = self.build_family_context_features(
            terminal,
            probabilities_df,
            markets_df[['market_id', 'family_id']],
            time_col='cutoff_timestamp_utc',
            prob_col='market_price_baseline',
        )
        terminal = terminal.merge(context, on=['market_id', 'cutoff_timestamp_utc'], how='left')
        return self.add_domain_dummies(terminal)

    def build_repricing_panel(self, markets_df: pd.DataFrame, probabilities_df: pd.DataFrame, *, shock_table: pd.DataFrame | None = None, shock_max_age: str | pd.Timedelta | None = None) -> pd.DataFrame:
        """Build the repricing panel and optionally merge external shocks."""
        repricing = build_repricing_dataset(
            markets_df,
            probabilities_df,
            future_horizon_hours=self.repricing_future_hours,
            lookback_hours=self.repricing_lookback_hours,
            sample_every_hours=self.repricing_sample_every_hours,
            move_threshold=self.repricing_move_threshold,
        )
        repricing = repricing.merge(markets_df[['market_id', 'primary_domain', 'question', 'tag_labels', 'family_id']], on='market_id', how='left')
        if shock_table is not None:
            repricing = asof_join_covariates(repricing, shock_table, base_time_col='timestamp_utc', max_age=shock_max_age)
        return self.add_domain_dummies(repricing)


## Локальный класс для внешних shocks

Этот блок остаётся notebook-local, потому что определение shock пока экспериментальное. Здесь мы всего лишь переводим BTC/ETH ряды в `return / z-score / shock flag`.


In [ ]:
@dataclass(frozen=True)
class ExternalShockBuilder:
    """Build BTC/ETH external shock tables from saved covariates."""

    z_threshold: float = 2.0
    std_window: int = 288

    def build(self, path: str | Path) -> pd.DataFrame:
        """Convert BTC/ETH external series into returns, z-scores, and shock flags."""
        covariates = load_external_covariates(path)
        wide = pivot_covariates_to_wide(covariates, value_col='value').sort_values('timestamp_utc').reset_index(drop=True)
        out = wide[['timestamp_utc']].copy()
        for series_id in ('btc_usd', 'eth_usd'):
            series = pd.to_numeric(wide.get(series_id), errors='coerce')
            returns = series.pct_change()
            sigma = returns.rolling(self.std_window, min_periods=max(24, self.std_window // 6)).std()
            z_score = returns / sigma.replace(0.0, np.nan)
            out[f'{series_id}_ret'] = returns
            out[f'{series_id}_z'] = z_score
            out[f'{series_id}_shock'] = (z_score.abs() >= self.z_threshold).astype(float)
        out['any_external_shock'] = out[['btc_usd_shock', 'eth_usd_shock']].max(axis=1)
        out['btc_or_eth_shock'] = out['any_external_shock']
        return out


## Локальный класс для retrieval-conditioned context

Это самая исследовательская часть, поэтому она тоже остаётся в notebook. Здесь живёт выбор retrieval universe, nearest-neighbor mapping и агрегирование contemporaneous neighbor state.


In [ ]:
@dataclass(frozen=True)
class RetrievedContextBuilder:
    """Retrieve related markets and summarize their contemporaneous state."""

    min_probability_rows: int = 288
    max_markets_per_domain: int = 90
    top_k: int = 5

    def prepare_market_universe(self, markets_df: pd.DataFrame, domains: tuple[str, ...]) -> pd.DataFrame:
        """Select a bounded retrieval universe from the full market table."""
        frames = []
        for domain in domains:
            frame = markets_df.loc[markets_df['domain'] == domain].copy()
            frame = frame.loc[frame['probability_rows'].fillna(0) >= self.min_probability_rows]
            if self.max_markets_per_domain is not None:
                frame = frame.sort_values('volume_num', ascending=False).head(self.max_markets_per_domain)
            frames.append(frame)
        return pd.concat(frames, ignore_index=True)

    def build_neighbor_map(self, retrieval_markets: pd.DataFrame) -> tuple[dict[str, list[str]], pd.DataFrame, np.ndarray]:
        """Build a text-and-tag-based nearest-neighbor map over markets."""
        work = retrieval_markets[['market_id', 'question', 'description', 'tag_labels', 'primary_domain', 'created_at', 'end_date', 'family_id']].copy()
        work['text_blob'] = work['question'].fillna('') + ' ' + work['description'].fillna('') + ' ' + work['tag_labels'].fillna('')
        vectorizer = TfidfVectorizer(min_df=2, max_features=5000, ngram_range=(1, 2), stop_words='english')
        tfidf = vectorizer.fit_transform(work['text_blob'])
        text_cos = cosine_similarity(tfidf)

        neighbor_map = {}
        detail_rows = []
        for i, row in work.reset_index(drop=True).iterrows():
            scores = []
            for j, row2 in work.reset_index(drop=True).iterrows():
                if i == j:
                    continue
                tag_overlap = len(set(parse_listish(row['tag_labels'])) & set(parse_listish(row2['tag_labels'])))
                score = 0.75 * float(text_cos[i, j]) + 0.25 * float(tag_overlap > 0)
                scores.append((score, row2['market_id'], row2['question']))
            scores.sort(reverse=True)
            top_neighbors = scores[: self.top_k]
            neighbor_map[row['market_id']] = [market_id for _, market_id, _ in top_neighbors]
            if i < 5:
                for rank, (score, neighbor_id, neighbor_question) in enumerate(top_neighbors, start=1):
                    detail_rows.append({
                        'query_market_id': row['market_id'],
                        'query_question': row['question'],
                        'rank': rank,
                        'neighbor_market_id': neighbor_id,
                        'score': score,
                        'neighbor_question': neighbor_question,
                    })
        return neighbor_map, pd.DataFrame(detail_rows), text_cos

    def build_context_features(self, panel_df: pd.DataFrame, probabilities_df: pd.DataFrame, neighbor_map: dict[str, list[str]], *, time_col: str = 'timestamp_utc') -> pd.DataFrame:
        """Summarize contemporaneous neighbor state into a compact feature block."""
        rows = []
        for row in panel_df[['market_id', time_col]].itertuples(index=False):
            cutoff = getattr(row, time_col)
            neighbors = neighbor_map.get(row.market_id, [])
            probs = []
            trades = []
            for neighbor_id in neighbors:
                snapshot = MarketPanelBuilder.latest_snapshot_before(probabilities_df, neighbor_id, cutoff)
                if snapshot is None:
                    continue
                probs.append(float(snapshot['yes_probability']))
                trades.append(float(snapshot.get('observed_trade', 0.0)))
            rows.append({
                'market_id': row.market_id,
                time_col: cutoff,
                'retrieved_count': float(len(probs)),
                'retrieved_prob_mean': float(np.mean(probs)) if probs else np.nan,
                'retrieved_prob_std': float(np.std(probs)) if probs else np.nan,
                'retrieved_prob_gap': float(np.max(probs) - np.min(probs)) if probs else np.nan,
                'retrieved_trade_share': float(np.mean(trades)) if trades else np.nan,
            })
        return pd.DataFrame(rows)


## Сбор основных панелей: terminal и repricing

Сейчас мы строим два главных датасета.

- `terminal`: snapshot рынка за фиксированное время до resolution и финальный outcome.
- `repricing`: snapshot рынка в момент `t` и сдвиг вероятности за следующие 24 часа.

На что смотреть в выводе ниже: сколько строк получилось, сколько уникальных рынков покрыто и какие домены реально попали в анализ.

In [3]:
DATASET_ARTEFACT_DIR = REPO_ROOT / 'research_notebooks' / 'running_artefacts'

dataset = PolymarketDataset.from_parquet(DATASET_ARTEFACT_DIR)
markets = panel_builder.prepare_markets(dataset.markets)
probabilities = panel_builder.prepare_probabilities(dataset, markets)

terminal = panel_builder.build_terminal_panel(markets, probabilities)
shock_table = shock_builder.build(EXTERNAL_PATH)
repricing = panel_builder.build_repricing_panel(
    markets,
    probabilities,
    shock_table=shock_table,
    shock_max_age=SHOCK_MAX_AGE,
)
for col in ['btc_usd_ret', 'btc_usd_z', 'btc_usd_shock', 'eth_usd_ret', 'eth_usd_z', 'eth_usd_shock', 'any_external_shock', 'btc_or_eth_shock']:
    repricing[col] = pd.to_numeric(repricing[col], errors='coerce').fillna(0.0)


Terminal rows: 966 Markets: 342
Repricing rows: 80755 Markets: 453
Domains used: politics, geopolitics, technology, finance_economy
Polymarket crypto domain is excluded by design; BTC/ETH are kept only as external signals.


## Покрытие данных и ключевые колонки

Эта секция нужна, чтобы сразу увидеть, **чем мы вообще оперируем**.

Важные таблицы ниже:

- `markets_summary`: сколько рынков у нас по доменам и насколько длинные их истории;
- `terminal_summary`: сколько snapshot'ов у terminal-задачи по доменам и горизонтам;
- `repricing_summary`: насколько часто large repricing вообще случается;
- `key_columns_guide`: мини-словарь самых важных колонок, чтобы дальше не читать код вслепую.

Особенно смотри на колонки `rows`, `markets`, `repricing_rate`, `market_abs_error`, `future_move`.

In [4]:

markets_summary = markets.assign(market_age_days=(markets['end_date'] - markets['created_at']).dt.total_seconds() / 86400.0).groupby('primary_domain', dropna=False).agg(
    markets=('market_id', 'nunique'),
    mean_market_age_days=('market_age_days', 'mean'),
    mean_probability_rows=('probability_rows', 'mean'),
    mean_trade_rows=('trade_rows', 'mean'),
).reset_index().sort_values('markets', ascending=False)

display(markets_summary)

terminal_summary = terminal.groupby(['primary_domain', 'horizon_name'], dropna=False).agg(
    rows=('market_id', 'size'),
    markets=('market_id', 'nunique'),
    positive_rate=('target', 'mean'),
    mean_abs_error=('market_abs_error', 'mean'),
).reset_index()
repricing_summary = repricing.groupby('primary_domain', dropna=False).agg(
    rows=('market_id', 'size'),
    markets=('market_id', 'nunique'),
    repricing_rate=('target', 'mean'),
    mean_abs_future_move=('future_move', lambda s: np.mean(np.abs(s))),
).reset_index()
display(terminal_summary)
display(repricing_summary)

key_columns_guide = pd.DataFrame([
    {'panel': 'terminal', 'column': 'market_price_baseline', 'meaning': 'текущая рыночная вероятность на момент cutoff; это главный baseline'},
    {'panel': 'terminal', 'column': 'market_abs_error', 'meaning': 'насколько текущая вероятность далека от финального исхода'},
    {'panel': 'terminal', 'column': 'family_prob_gap', 'meaning': 'разброс вероятностей среди родственных рынков'},
    {'panel': 'repricing', 'column': 'future_move', 'meaning': 'сдвиг вероятности за следующие 24 часа'},
    {'panel': 'repricing', 'column': 'recent_volatility', 'meaning': 'локальная нестабильность рынка до текущего момента'},
    {'panel': 'repricing', 'column': 'btc_or_eth_shock', 'meaning': 'внешний shock-флаг по BTC/ETH на момент snapshot'},
])
display(key_columns_guide)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=terminal_summary, x='rows', y='primary_domain', hue='horizon_name', ax=axes[0], palette='crest')
axes[0].set_title('Покрытие terminal panel по доменам и горизонтам')
axes[0].set_xlabel('Число строк')
axes[0].set_ylabel('')

sns.barplot(data=repricing_summary, x='repricing_rate', y='primary_domain', ax=axes[1], palette='flare')
axes[1].set_title('Доля large repricing по доменам')
axes[1].set_xlabel('Частота repricing')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

terminal_key_columns = terminal[[
    'market_id', 'primary_domain', 'horizon_name', 'market_price_baseline', 'current_yes_probability', 'market_abs_error', 'hours_to_resolution', 'family_related_count', 'family_prob_gap'
]].head(8)
repricing_key_columns = repricing[[
    'market_id', 'primary_domain', 'timestamp_utc', 'current_yes_probability', 'future_move', 'target', 'recent_volatility', 'confidence_margin', 'btc_or_eth_shock'
]].head(8)
display(terminal_key_columns)
display(repricing_key_columns)


## 1. Terminal forecasting как reference task

Это не главный шанс на paper contribution, а **reference task**. Его задача — показать, насколько силён текущий рынок сам по себе, и где вообще остаётся пространство для модели.

Сначала посмотрим не на leaderboard, а на структуру сложности задачи.

Важные колонки в следующих таблицах:

- `near_extreme_share`: доля snapshot'ов, где цена уже почти выродилась к `0` или `1`;
- `mean_abs_error`: насколько в среднем текущий рынок всё ещё ошибается;
- `mean_confidence_margin`: насколько рынок вообще уверен;
- `bad_state_rate`: доля действительно тяжёлых состояний внутри uncertainty buckets.

In [5]:

terminal_diag = terminal.copy()
terminal_diag['near_extreme_price'] = ((terminal_diag['market_price_baseline'] <= 0.05) | (terminal_diag['market_price_baseline'] >= 0.95)).astype(int)
terminal_diag['uncertainty_bucket'] = pd.cut(
    terminal_diag['confidence_margin'],
    bins=[-1e-9, 0.10, 0.20, 0.35, 0.50],
    labels=['0.00-0.10', '0.10-0.20', '0.20-0.35', '0.35-0.50'],
)

terminal_horizon_diag = terminal_diag.groupby('horizon_name', dropna=False).agg(
    rows=('market_id', 'size'),
    markets=('market_id', 'nunique'),
    near_extreme_share=('near_extreme_price', 'mean'),
    mean_abs_error=('market_abs_error', 'mean'),
    mean_confidence_margin=('confidence_margin', 'mean'),
).reset_index()

display(terminal_horizon_diag)

terminal_uncertainty_diag = terminal_diag.groupby(['horizon_name', 'uncertainty_bucket'], dropna=False).agg(
    rows=('market_id', 'size'),
    bad_state_rate=('market_abs_error', lambda s: np.mean(s >= 0.25)),
    mean_abs_error=('market_abs_error', 'mean'),
).reset_index()

display(terminal_uncertainty_diag)


Теперь уже можно смотреть на модели. Ниже важны не только абсолютные метрики, но и **дельты относительно `market_price`**.

На какие колонки смотреть:

- `log_loss`: главный quality metric для вероятностного прогноза;
- `delta_log_loss_vs_market`: насколько модель хуже или лучше рынка;
- `delta_rawA_vs_market` и `delta_rawAB_vs_market`: видно, помогает ли local/context model хотя бы на отдельных горизонтах.

In [6]:
local_cols = [
    'current_yes_probability', 'confidence_margin', 'snapshot_staleness_hours', 'observed_trade_now', 'trade_count_now', 'total_size_now', 'last_trade_price_now',
    'lookback_24h_yes_probability_change', 'lookback_24h_volatility', 'lookback_168h_yes_probability_change', 'lookback_168h_volatility',
    'hours_to_resolution', 'market_age_hours', 'life_progress', 'horizon_hours',
] + [col for col in terminal.columns if col.startswith('domain_')]
context_cols = ['family_related_count', 'family_prob_mean', 'family_prob_gap', 'family_vs_market_gap']

terminal_rows = []
for train_df, test_df, meta in rolling_time_splits(terminal, time_col='end_date', n_splits=4, min_train_fraction=0.5):
    y_test = test_df['target'].astype(int).to_numpy()
    base_prob = clipped(test_df['market_price_baseline'])
    terminal_rows.append({'variant': 'market_price', 'fold': meta['fold'], 'log_loss': log_loss(y_test, base_prob), 'brier': brier_score_loss(y_test, base_prob), 'roc_auc': safe_auc(y_test, base_prob)})
    for variant_name, cols in [('raw(A)', local_cols), ('raw(A)+raw(B)', local_cols + context_cols)]:
        model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=2000))])
        model.fit(train_df[cols], train_df['target'].astype(int).to_numpy())
        p_test = clipped(model.predict_proba(test_df[cols])[:, 1])
        terminal_rows.append({'variant': variant_name, 'fold': meta['fold'], 'log_loss': log_loss(y_test, p_test), 'brier': brier_score_loss(y_test, p_test), 'roc_auc': safe_auc(y_test, p_test)})

terminal_metrics = pd.DataFrame(terminal_rows)
terminal_table = terminal_metrics.groupby('variant', dropna=False)[['log_loss', 'brier', 'roc_auc']].mean().sort_values('log_loss').reset_index()
display(terminal_table)

terminal_delta = terminal_table.copy()
base_log_loss = float(terminal_delta.loc[terminal_delta['variant'] == 'market_price', 'log_loss'].iloc[0])
base_auc = float(terminal_delta.loc[terminal_delta['variant'] == 'market_price', 'roc_auc'].iloc[0])
terminal_delta['delta_log_loss_vs_market'] = terminal_delta['log_loss'] - base_log_loss
terminal_delta['delta_roc_auc_vs_market'] = terminal_delta['roc_auc'] - base_auc
display(terminal_delta)

terminal_by_horizon_rows = []
for train_df, test_df, meta in rolling_time_splits(terminal, time_col='end_date', n_splits=4, min_train_fraction=0.5):
    y_test = test_df['target'].astype(int).to_numpy()
    for horizon_name, horizon_df in test_df.groupby('horizon_name', dropna=False):
        y_h = horizon_df['target'].astype(int).to_numpy()
        if len(y_h) < 5:
            continue
        base_prob = clipped(horizon_df['market_price_baseline'])
        terminal_by_horizon_rows.append({
            'variant': 'market_price',
            'horizon_name': horizon_name,
            'log_loss': log_loss(y_h, base_prob),
            'roc_auc': safe_auc(y_h, base_prob),
            'rows': len(horizon_df),
        })
        for variant_name, cols in [('raw(A)', local_cols), ('raw(A)+raw(B)', local_cols + context_cols)]:
            model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=2000))])
            model.fit(train_df[cols], train_df['target'].astype(int).to_numpy())
            p_h = clipped(model.predict_proba(horizon_df[cols])[:, 1])
            terminal_by_horizon_rows.append({
                'variant': variant_name,
                'horizon_name': horizon_name,
                'log_loss': log_loss(y_h, p_h),
                'roc_auc': safe_auc(y_h, p_h),
                'rows': len(horizon_df),
            })

terminal_by_horizon = pd.DataFrame(terminal_by_horizon_rows)
terminal_horizon_pivot = terminal_by_horizon.pivot_table(index='horizon_name', columns='variant', values='log_loss', aggfunc='mean').reset_index()
if {'market_price', 'raw(A)', 'raw(A)+raw(B)'} <= set(terminal_horizon_pivot.columns):
    terminal_horizon_pivot['delta_rawA_vs_market'] = terminal_horizon_pivot['raw(A)'] - terminal_horizon_pivot['market_price']
    terminal_horizon_pivot['delta_rawAB_vs_market'] = terminal_horizon_pivot['raw(A)+raw(B)'] - terminal_horizon_pivot['market_price']
display(terminal_horizon_pivot)


График нужен не для leaderboard, а для визуального вывода: насколько далеко простые модели от сильного market baseline. Если разрыв большой и устойчивый, это аргумент, что terminal forecasting не должен быть единственным headline result.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=terminal_table, x='log_loss', y='variant', ax=axes[0], palette='crest')
axes[0].set_title('Terminal forecasting: средний log loss')
axes[0].set_xlabel('Ниже лучше')
axes[0].set_ylabel('')

sns.barplot(data=terminal_table.sort_values('roc_auc', ascending=False), x='roc_auc', y='variant', ax=axes[1], palette='flare')
axes[1].set_title('Terminal forecasting: средний ROC-AUC')
axes[1].set_xlabel('Выше лучше')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 2. Trustworthiness как meta-prediction

Здесь вопрос уже другой: **не что случится**, а **можно ли доверять текущей рыночной вероятности**.

Сначала полезно посмотреть, как bad states распределяются по уровням уверенности рынка. В следующей таблице главные колонки:

- `bad_state_rate`: как часто уверенный рынок всё же ошибается сильно;
- `mean_abs_error`: средняя фактическая ошибка;
- `mean_family_gap`: насколько bad states связаны с конфликтом среди родственных рынков.

In [8]:

trust_diag = terminal.copy()
trust_diag['bad_state'] = (trust_diag['market_abs_error'] >= 0.25).astype(int)
trust_diag['confidence_bin'] = pd.qcut(trust_diag['confidence_margin'], q=5, duplicates='drop')
trust_confidence_table = trust_diag.groupby('confidence_bin', dropna=False).agg(
    rows=('market_id', 'size'),
    bad_state_rate=('bad_state', 'mean'),
    mean_abs_error=('market_abs_error', 'mean'),
    mean_family_gap=('family_prob_gap', 'mean'),
).reset_index()
display(trust_confidence_table)


Теперь сравниваем два trust-подхода: простой `confidence_margin` и обучаемую модель.

Ключевые колонки:

- `average_precision`: насколько хорошо risk-score находит плохие состояния;
- `roc_auc`: общее качество ранжирования;
- `bad_state_rate` и `mean_abs_error` в retained subset: отвечают на вопрос, полезна ли trust-модель для selective prediction.

In [9]:
trust_df = terminal.copy()
trust_df['bad_state'] = (trust_df['market_abs_error'] >= 0.25).astype(int)
trust_features = local_cols + context_cols

trust_rows = []
coverage_rows = []
coverage_grid = np.linspace(0.1, 1.0, 10)

for train_df, test_df, meta in rolling_time_splits(trust_df, time_col='end_date', n_splits=4, min_train_fraction=0.5):
    y_test = test_df['bad_state'].astype(int).to_numpy()
    baseline_risk = 1.0 - 2.0 * test_df['confidence_margin'].to_numpy()
    trust_rows.append({'variant': 'confidence_margin', 'fold': meta['fold'], 'average_precision': safe_ap(y_test, baseline_risk), 'roc_auc': safe_auc(y_test, baseline_risk)})
    model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))])
    model.fit(train_df[trust_features], train_df['bad_state'].astype(int).to_numpy())
    learned_risk = model.predict_proba(test_df[trust_features])[:, 1]
    trust_rows.append({'variant': 'learned_trust', 'fold': meta['fold'], 'average_precision': safe_ap(y_test, learned_risk), 'roc_auc': safe_auc(y_test, learned_risk)})
    for variant_name, risk_score in [('confidence_margin', baseline_risk), ('learned_trust', learned_risk)]:
        order = np.argsort(risk_score)
        kept_df = test_df.iloc[order].copy()
        for coverage in coverage_grid:
            keep_n = max(1, int(len(kept_df) * coverage))
            kept = kept_df.iloc[:keep_n]
            coverage_rows.append({'variant': variant_name, 'fold': meta['fold'], 'coverage': coverage, 'bad_state_rate': kept['bad_state'].mean(), 'mean_abs_error': kept['market_abs_error'].mean()})

trust_metrics = pd.DataFrame(trust_rows)
trust_curve = pd.DataFrame(coverage_rows)
display(trust_metrics.groupby('variant', dropna=False)[['average_precision', 'roc_auc']].mean().reset_index())

trust_slice = trust_df.groupby(['primary_domain', 'horizon_name'], dropna=False).agg(
    rows=('market_id', 'size'),
    bad_state_rate=('bad_state', 'mean'),
    mean_abs_error=('market_abs_error', 'mean'),
    mean_confidence_margin=('confidence_margin', 'mean'),
).reset_index()
display(trust_slice)

Два графика ниже — это уже practical view на trust. По оси `coverage` показано, какую долю snapshot'ов мы оставляем. Если кривая ниже, значит модель лучше умеет отбрасывать ненадёжные состояния.

In [10]:
curve_summary = trust_curve.groupby(['variant', 'coverage'], dropna=False)[['bad_state_rate', 'mean_abs_error']].mean().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.lineplot(data=curve_summary, x='coverage', y='bad_state_rate', hue='variant', marker='o', ax=axes[0])
axes[0].set_title('Selective trust: доля плохих состояний после фильтрации')
axes[0].set_xlabel('Оставленная доля снапшотов')
axes[0].set_ylabel('Ниже лучше')

sns.lineplot(data=curve_summary, x='coverage', y='mean_abs_error', hue='variant', marker='o', ax=axes[1])
axes[1].set_title('Selective trust: средняя абсолютная ошибка после фильтрации')
axes[1].set_xlabel('Оставленная доля снапшотов')
axes[1].set_ylabel('Ниже лучше')
plt.tight_layout()
plt.show()

## 3. Repricing как основной динамический benchmark

Вот здесь начинается более живая scientific story. Мы не спрашиваем только про terminal outcome, а пытаемся понять: **вот-вот ли рынок пересмотрит своё belief state**.

Сначала — descriptive диагностика. Важные колонки:

- `repricing_rate`: как часто большие сдвиги вообще случаются;
- `mean_abs_future_move`: средний размер будущего сдвига;
- `mean_confidence_margin` и `mean_recent_volatility`: какие режимы чаще ведут к repricing.

In [11]:

repricing_diag = repricing.copy()
repricing_diag['volatility_bin'] = pd.qcut(repricing_diag['recent_volatility'], q=5, duplicates='drop')
repricing_diag['confidence_bin'] = pd.qcut(repricing_diag['confidence_margin'], q=5, duplicates='drop')

volatility_table = repricing_diag.groupby('volatility_bin', dropna=False).agg(
    rows=('market_id', 'size'),
    repricing_rate=('target', 'mean'),
    mean_abs_future_move=('future_move', lambda s: np.mean(np.abs(s))),
    mean_confidence_margin=('confidence_margin', 'mean'),
).reset_index()
confidence_table = repricing_diag.groupby('confidence_bin', dropna=False).agg(
    rows=('market_id', 'size'),
    repricing_rate=('target', 'mean'),
    mean_abs_future_move=('future_move', lambda s: np.mean(np.abs(s))),
    mean_recent_volatility=('recent_volatility', 'mean'),
).reset_index()

display(volatility_table)
display(confidence_table)


Теперь смотрим на первые predictive модели. В таблицах ниже самый важный вопрос такой: дают ли внешние shocks что-то сверх обычной market microstructure.

Смотри прежде всего на:

- `average_precision`: основной metric для редкого события;
- `delta_ap_vs_microstructure`: есть ли польза сверх базовой модели;
- `shock_examples`: не leaderboard, а sanity check, какие реальные сильные эпизоды попадают в данные.

In [12]:
base_cols = [
    'current_yes_probability', 'confidence_margin', 'hours_to_resolution', 'life_progress',
    'recent_abs_move_mean', 'recent_abs_move_max', 'recent_volatility', 'recent_directional_move',
    'observed_trade_share', 'trade_count_sum', 'total_size_sum',
] + [col for col in repricing.columns if col.startswith('domain_')]
shock_cols = ['btc_usd_ret', 'btc_usd_z', 'btc_usd_shock', 'eth_usd_ret', 'eth_usd_z', 'eth_usd_shock', 'any_external_shock']

shock_descriptive = repricing.groupby('btc_or_eth_shock', dropna=False).agg(rows=('market_id', 'size'), markets=('market_id', 'nunique'), repricing_rate=('target', 'mean'), mean_abs_future_move=('future_move', lambda s: np.mean(np.abs(s)))).reset_index()
display(shock_descriptive)

repricing_rows = []
for train_df, test_df, meta in rolling_time_splits(repricing, time_col='timestamp_utc', n_splits=4, min_train_fraction=0.5):
    y_test = test_df['target'].astype(int).to_numpy()
    for variant_name, cols in [('microstructure', base_cols), ('microstructure+shocks', base_cols + shock_cols), ('shocks_only', shock_cols + [col for col in repricing.columns if col.startswith('domain_')])]:
        model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))])
        model.fit(train_df[cols], train_df['target'].astype(int).to_numpy())
        p_test = clipped(model.predict_proba(test_df[cols])[:, 1])
        repricing_rows.append({'variant': variant_name, 'fold': meta['fold'], 'average_precision': safe_ap(y_test, p_test), 'roc_auc': safe_auc(y_test, p_test), 'log_loss': log_loss(y_test, p_test)})

repricing_metrics = pd.DataFrame(repricing_rows)
repricing_table = repricing_metrics.groupby('variant', dropna=False)[['average_precision', 'roc_auc', 'log_loss']].mean().sort_values('average_precision', ascending=False).reset_index()
display(repricing_table)

repricing_delta = repricing_table.copy()
base_ap = float(repricing_delta.loc[repricing_delta['variant'] == 'microstructure', 'average_precision'].iloc[0])
base_auc = float(repricing_delta.loc[repricing_delta['variant'] == 'microstructure', 'roc_auc'].iloc[0])
repricing_delta['delta_ap_vs_microstructure'] = repricing_delta['average_precision'] - base_ap
repricing_delta['delta_auc_vs_microstructure'] = repricing_delta['roc_auc'] - base_auc
display(repricing_delta)

shock_examples = repricing.loc[repricing['btc_or_eth_shock'] >= 1.0, ['timestamp_utc', 'primary_domain', 'question', 'current_yes_probability', 'future_move', 'target', 'btc_usd_z', 'eth_usd_z']].assign(abs_future_move=lambda x: x['future_move'].abs()).sort_values('abs_future_move', ascending=False).head(10)
display(shock_examples)

Этот блок графиков нужен, чтобы разделить descriptive и predictive story. Слева видно, есть ли хоть грубая реактивность доменов на shocks. Справа — достаточно ли этого для реального repricing benchmark.

In [13]:
domain_response = repricing.groupby(['primary_domain', 'btc_or_eth_shock'], dropna=False).agg(rows=('market_id', 'size'), repricing_rate=('target', 'mean'), mean_abs_future_move=('future_move', lambda s: np.mean(np.abs(s)))).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=domain_response, x='repricing_rate', y='primary_domain', hue='btc_or_eth_shock', ax=axes[0], palette='crest')
axes[0].set_title('Частота repricing по доменам и shock-состоянию')
axes[0].set_xlabel('Выше = домен реагирует сильнее')
axes[0].set_ylabel('')

sns.barplot(data=repricing_table, x='average_precision', y='variant', ax=axes[1], palette='flare')
axes[1].set_title('Repricing prediction: средний average precision')
axes[1].set_xlabel('Выше лучше')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 4. Retrieval-conditioned context: первый реальный context result

Здесь мы впервые делаем шаг к будущему encoder'у. Вместо того чтобы просто дать модели плоскую таблицу, мы сначала строим **retrieved context**: находим похожие рынки и суммируем их contemporaneous state.

Ниже важны две вещи:

- `retrieved_summary`: сколько соседей реально находится и насколько они разнообразны;
- `neighbor_detail_rows`: sanity check на уровне текста — действительно ли retrieval находит похожие вопросы, а не случайный шум.

Ключевые колонки:

- `retrieved_count`: сколько соседей реально доступно на момент snapshot;
- `retrieved_prob_gap`: насколько мнения retrieved neighbors между собой расходятся;
- `retrieved_trade_share`: насколько эти соседи вообще активны.

In [14]:
retrieval_markets = retrieval_builder.prepare_market_universe(markets, DOMAINS)
neighbor_map, neighbor_examples, text_cos = retrieval_builder.build_neighbor_map(retrieval_markets)
retrieved_context = retrieval_builder.build_context_features(repricing, probabilities, neighbor_map, time_col='timestamp_utc')
repricing_v2 = repricing.merge(retrieved_context, on=['market_id', 'timestamp_utc'], how='left')
for col in RETRIEVED_CONTEXT_COLS:
    repricing_v2[col] = pd.to_numeric(repricing_v2[col], errors='coerce').fillna(0.0)

retrieved_summary = repricing_v2[RETRIEVED_CONTEXT_COLS].agg(['mean', 'median']).T.reset_index().rename(columns={'index': 'feature'})
display(retrieved_summary)
display(neighbor_examples)


Следующая таблица — одна из самых важных во всём notebook. Она уже показывает не просто similarity-диагностику, а **downstream usefulness** retrieved context.

Смотри на:

- `average_precision`: выигрыш или проигрыш по сравнению с `microstructure`;
- `delta_log_loss_vs_microstructure`: меняется ли калибровка, даже если AP меняется мало;
- `delta_ap_from_context` по доменам: context помогает не везде одинаково, и это важная научная часть истории.

In [15]:
repricing_v2_rows = []
pred_rows = []
for train_df, test_df, meta in rolling_time_splits(repricing_v2, time_col='timestamp_utc', n_splits=4, min_train_fraction=0.5):
    y_test = test_df['target'].astype(int).to_numpy()
    for variant_name, cols in [
        ('microstructure', base_cols),
        ('microstructure+retrieved_context', base_cols + RETRIEVED_CONTEXT_COLS),
        ('retrieved_context_only', RETRIEVED_CONTEXT_COLS + [col for col in repricing_v2.columns if col.startswith('domain_')]),
    ]:
        model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))])
        model.fit(train_df[cols], train_df['target'].astype(int).to_numpy())
        p_test = clipped(model.predict_proba(test_df[cols])[:, 1])
        repricing_v2_rows.append({'variant': variant_name, 'fold': meta['fold'], 'average_precision': safe_ap(y_test, p_test), 'roc_auc': safe_auc(y_test, p_test), 'log_loss': log_loss(y_test, p_test)})
        if variant_name in {'microstructure', 'microstructure+retrieved_context'}:
            tmp = test_df[['primary_domain', 'target', 'confidence_margin']].copy()
            tmp['variant'] = variant_name
            tmp['pred'] = p_test
            pred_rows.append(tmp)

repricing_v2_metrics = pd.DataFrame(repricing_v2_rows)
repricing_v2_table = repricing_v2_metrics.groupby('variant', dropna=False)[['average_precision', 'roc_auc', 'log_loss']].mean().sort_values('average_precision', ascending=False).reset_index()
display(repricing_v2_table)

repricing_v2_delta = repricing_v2_table.copy()
base_ap = float(repricing_v2_delta.loc[repricing_v2_delta['variant'] == 'microstructure', 'average_precision'].iloc[0])
base_auc = float(repricing_v2_delta.loc[repricing_v2_delta['variant'] == 'microstructure', 'roc_auc'].iloc[0])
base_ll = float(repricing_v2_delta.loc[repricing_v2_delta['variant'] == 'microstructure', 'log_loss'].iloc[0])
repricing_v2_delta['delta_ap_vs_microstructure'] = repricing_v2_delta['average_precision'] - base_ap
repricing_v2_delta['delta_auc_vs_microstructure'] = repricing_v2_delta['roc_auc'] - base_auc
repricing_v2_delta['delta_log_loss_vs_microstructure'] = repricing_v2_delta['log_loss'] - base_ll
display(repricing_v2_delta)

pred_df = pd.concat(pred_rows, ignore_index=True)
domain_uplift = pred_df.groupby(['primary_domain', 'variant'], dropna=False).apply(lambda g: pd.Series({'average_precision': safe_ap(g['target'], g['pred']), 'roc_auc': safe_auc(g['target'], g['pred']), 'rows': len(g)}), include_groups=False).reset_index()
domain_compare = domain_uplift.pivot_table(index='primary_domain', columns='variant', values='average_precision', aggfunc='mean').reset_index()
if {'microstructure', 'microstructure+retrieved_context'} <= set(domain_compare.columns):
    domain_compare['delta_ap_from_context'] = domain_compare['microstructure+retrieved_context'] - domain_compare['microstructure']
display(domain_compare.sort_values('delta_ap_from_context', ascending=False))


Графики ниже нужны, чтобы быстро увидеть две вещи: общий выигрыш от retrieved context и его гетерогенность по доменам. Если прирост точечный, это не делает результат слабым; наоборот, это говорит, что context signal **селективен**, а не универсален.

In [16]:
pred_df['uncertain_state'] = (pred_df['confidence_margin'] <= 0.20).astype(int)
uncertainty_uplift = pred_df.groupby(['uncertain_state', 'variant'], dropna=False).apply(lambda g: pd.Series({'average_precision': safe_ap(g['target'], g['pred']), 'roc_auc': safe_auc(g['target'], g['pred']), 'rows': len(g)}), include_groups=False).reset_index()

display(domain_uplift.sort_values(['primary_domain', 'average_precision'], ascending=[True, False]))
display(uncertainty_uplift.sort_values(['uncertain_state', 'average_precision'], ascending=[True, False]))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=repricing_v2_table, x='average_precision', y='variant', ax=axes[0], palette='crest')
axes[0].set_title('Repricing с retrieved context: общий AP')
axes[0].set_xlabel('Выше лучше')
axes[0].set_ylabel('')

sns.barplot(data=domain_uplift, x='average_precision', y='primary_domain', hue='variant', ax=axes[1], palette='flare')
axes[1].set_title('Гетерогенность выигрыша от retrieved context по доменам')
axes[1].set_xlabel('Average precision')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()


## 5. Почему retrieval вообще правдоподобен

До этого retrieval уже помогал downstream, но reviewer справедливо спросит: почему мы вообще считаем этот retrieval осмысленным?

В следующих таблицах мы проверяем, насколько simple closeness scores действительно восстанавливают weak labels вроде `same_family` и `same_domain`.

Главные колонки:

- `roc_auc`: насколько score ранжирует релевантные пары выше нерелевантных;
- `average_precision`: особенно важен при редких `same_family` парах.

In [17]:
pair_rows = []
market_ids = retrieval_markets['market_id'].tolist()
for i, j in itertools.combinations(range(len(market_ids)), 2):
    row_a = retrieval_markets.iloc[i]
    row_b = retrieval_markets.iloc[j]
    pair_rows.append({
        'same_family': float(row_a['family_id'] == row_b['family_id']),
        'same_domain': float(row_a['primary_domain'] == row_b['primary_domain']),
        'text_cosine': float(text_cos[i, j]),
        'tag_jaccard': tag_jaccard(row_a['tag_labels'], row_b['tag_labels']),
        'combined_score': 0.7 * float(text_cos[i, j]) + 0.3 * tag_jaccard(row_a['tag_labels'], row_b['tag_labels']),
    })

pair_df = pd.DataFrame(pair_rows)
retrieval_eval_rows = []
for label_col in ['same_family', 'same_domain']:
    for score_col in ['text_cosine', 'tag_jaccard', 'combined_score']:
        retrieval_eval_rows.append({'label': label_col, 'score': score_col, 'roc_auc': safe_auc(pair_df[label_col], pair_df[score_col]), 'average_precision': safe_ap(pair_df[label_col], pair_df[score_col])})

retrieval_eval = pd.DataFrame(retrieval_eval_rows)
display(retrieval_eval.sort_values(['label', 'roc_auc'], ascending=[True, False]))

retrieval_eval_pivot = retrieval_eval.pivot(index='score', columns='label', values='roc_auc').reset_index()
display(retrieval_eval_pivot)



## Главные выводы из v2 после доработки narrative

1. `Terminal forecasting` нужен как reference task, но здесь рынок уже очень силён. Поэтому главная научная история не в том, чтобы в среднем победить `market_price`, а в том, чтобы понять, где контекст действительно добавляет информацию.
2. `Trustworthiness` оказался не просто вспомогательной задачей: уже простая `confidence_margin` хорошо ранжирует плохие состояния. Это делает задачу полезной как meta-prediction benchmark.
3. `Repricing` остаётся самым живым динамическим benchmark: именно там виден смысл cross-market context.
4. Простые внешние shocks `BTC/ETH` сами по себе слабы. Это важный negative result: одного внешнего сигнала недостаточно.
5. `Retrieved context` даёт небольшой, но честный выигрыш на repricing. Это и есть мост к следующему шагу: учить не flat tabular model, а target-conditioned context encoder.
